# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane CTR/ENGAGEMENT OPPORTUNITY SCORING as an ML task

My lane is a SCORING task.
The core problem is: pages are appearing in search but not getting engagement, and we need to see how they are actually doing.
So the model scores each page to show how well or how poorly it is performing.

Ranking is not the primary task type, it is the output of scoring.
Once the model produces a score for every page, we can order those scores from worst to best.
That ranked list is what tells a client which pages need to be fixed first.
So ranking is the application of the model's output, not the modeling task itself.

Classification could be used at the very end, as a secondary step.
Once pages are scored, we could split them into two simple categories, such as best scored versus worst scored, just to help decide which group to focus on first.
But that classification step depends entirely on the score already existing.
It cannot happen without scoring first, which is why scoring is the true underlying ML task, and both ranking and classification are things we build on top of it.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
### 2. Target or proxy

**What would you predict?**
A continuous engagement score for each page, showing how much a page is struggling to convert its search visibility into real engagement.

**Where does that label come from — observed outcome or a defined rule?**
A defined rule. This is a proxy, not an observed outcome. There is no column that directly records whether a page deserves attention. Instead, the label is built by engineering a composite feature from CTR and engagement rate, normalized to a common scale and combined into one engagement composite score. Pages with very low impressions are excluded, since low visibility pages cannot give a meaningful signal either way. A threshold is then set using the actual distribution of this composite score in the data, flagging the lowest percentile range as declining and the higher percentile range as performing fine. This makes the label a defined rule grounded in the real data, not an observed fact and not an arbitrary guess.

### 2. Target or proxy

**What would you predict?**
A continuous engagement score for each page, showing how much a page is struggling to convert its search visibility into real engagement.

**Where does that label come from — observed outcome or a defined rule?**
A defined rule. This is a proxy, not an observed outcome. There is no column that directly records whether a page deserves attention. Instead, the label is built by engineering a composite feature from CTR and engagement rate, normalized to a common scale and combined into one engagement composite score. Pages with very low impressions are excluded, since low visibility pages cannot give a meaningful signal either way. A threshold is then set using the actual distribution of this composite score in the data, flagging the lowest percentile range as declining and the higher percentile range as performing fine. This makes the label a defined rule grounded in the real data, not an observed fact and not an arbitrary guess.

In [12]:
import pandas as pd, numpy as np

url = "https://raw.githubusercontent.com/KhadijaHussnainMLEngineer/MLPipeline_MachineLearningFlyRankAI/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

print(df["ctr"].describe())
print(df["engagement_rate"].describe())

# Normalize each metric to 0-100 scale using min-max scaling
def normalize(col):
    return (col - col.min()) / (col.max() - col.min()) * 100

df["ctr_norm"] = normalize(df["ctr"])
df["engagement_norm"] = normalize(df["engagement_rate"])

# Composite: average of the two normalized signals
df["engagement_composite_score"] = (df["ctr_norm"] + df["engagement_norm"]) / 2

df[["ctr", "engagement_rate", "ctr_norm", "engagement_norm", "engagement_composite_score"]].head(10)

# Only trust the composite score where visibility is real
df["has_enough_visibility"] = df["impressions_90d"] >= 500  # pages actually worth judging
df["valid_composite_score"] = np.where(df["has_enough_visibility"], df["engagement_composite_score"], np.nan)
df["valid_composite_score"].describe()

# Use percentiles from the actual data to set the threshold
low_cutoff = df["valid_composite_score"].quantile(0.25)
high_cutoff = df["valid_composite_score"].quantile(0.75)

print(f"Declining cutoff (25th percentile): {low_cutoff:.3f}")
print(f"Performing fine cutoff (75th percentile): {high_cutoff:.3f}")

# Build the actual proxy label
df["engagement_proxy_label"] = np.where(
    df["valid_composite_score"] <= low_cutoff, "declining",
    np.where(df["valid_composite_score"] >= high_cutoff, "performing_fine", "middle")
)

# Show how many pages fall into each group
print(df["engagement_proxy_label"].value_counts())

(30000, 44)
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64
count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64
Declining cutoff (25th percentile): 0.050
Performing fine cutoff (75th percentile): 2.045
engagement_proxy_label
middle             21442
declining           4371
performing_fine     4187
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.